# 06 · Results summary and comparison with IncidentLens

This notebook collects the saved results of notebooks 03–05 into one table (it re-trains nothing).

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*(mismatched devices|pandas version 4).*")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
RESULTS = ROOT / "reports" / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

from rcalens.evaluate import evaluate
base = json.load(open(RESULTS / "baselines.json"))
ml = json.load(open(RESULTS / "ml_ranker.json"))
dl = json.load(open(RESULTS / "deep_learning.json")) if (RESULTS / "deep_learning.json").exists() else None
rows = {"random (expected)": base["overall"]["random (expected)"],
        f"best rule: {base['best_rule']}": base["overall"][base["best_rule"]],
        "XGBoost ranker (grouped CV, mean of 3)": ml["xgb_grouped_cv_mean"],
        "XGBoost ranker, unseen system": ml["xgb_leave_one_system_out"]["all"],
        "XGBoost ranker, unseen culprit service": ml["xgb_leave_one_rc_service_out"]["all"]}
if dl:
    rows["1D CNN (grouped CV)"] = dl["cnn_grouped_cv"]["all"]
    rows["XGBoost + CNN ensemble"] = dl["ensemble_xgb_cnn"]["all"]
    rows["1D CNN, unseen system"] = dl["cnn_leave_one_system_out"]["all"]
summary = pd.DataFrame(rows).T[["AC@1", "AC@3", "AC@5", "Avg@5", "MRR"]]
print("trace features:", ml["has_traces"], "| training device:", ml["device"])
summary.round(3)

In [ ]:
pd.DataFrame(ml["fault_type"]).T.round(3).rename_axis("fault-type classifier (6 classes)")

## Conclusions
*(Written after the full run on the training server.)*